In [0]:
#View emp data
dbutils.fs.head("/Volumes/databricks_practice/inputdb/empdata/old_emp_dta/employees.csv")



In [0]:
#convert csv to delta
from pyspark.sql.types import *
emp_schema=StructType([StructField("firstname",StringType(),True),StructField("Gender",StringType(),True)
                       ,StructField("joindate",DateType(),True),StructField("LOgintime",StringType(),True)
                       ,StructField("Salary",FloatType(),True),StructField("Bonuspercent",FloatType(),True)
                       ,StructField("Seniormagmt",BooleanType(),True),StructField("Team",StringType(),True) ])
df=spark.read.format("csv").schema(emp_schema).option("header","true").load("/Volumes/databricks_practice/inputdb/empdata/old_emp_dta/employees.csv")
df.show(3)
for col in df.columns:    df = df.withColumnRenamed(col, col.replace(' ', '_').replace('%', 'Percent').replace('-', '_'))
df.write.format("delta").mode("overwrite").save("/Volumes/databricks_practice/inputdb/empdata/old_emp_data_delta/")


In [0]:
#create a parmaeter Team
dbutils.widgets.text("team","","Team Name")

In [0]:
_team=dbutils.widgets.get("team")

In [0]:
print(_team)

In [0]:

emp_df=spark.read.load("/Volumes/databricks_practice/inputdb/empdata/old_emp_data_delta/")
emp_df.show(5)


In [0]:
#filter data based 
from pyspark.sql.functions import col,upper,lit
emp_filter=emp_df.filter(col("firstname").isNotNull() & (upper(col("Team")) == upper(lit(_team))))

In [0]:
emp_filter.show(5)

In [0]:
_count = emp_filter.count()
print(_count)
if _count > 0:
    emp_filter.write.mode("overwrite").saveAsTable(f"databricks_practice.outputdb.tblemp_team_{_team}")
    print(f"Data written for {_team}")
else:
    print(f"No Data written for {_team}")

In [0]:
#Exit status as no of records written 
dbutils.notebook.exit(_count)